## Task 1: Getting Started with AstraDB (DataStax)

1. Created an account on DataStax Astra DB.
2. Created a vector database named `astra_langchain`.
3. Generated the application token and copied the database API endpoint.
4. Saved the credentials securely for connecting LangChain with Astra DB.

Astra DB will be used to store document embeddings and retrieve relevant information for the PDF-based RAG application.

## Task 2: Connect LangChain with AstraDB

In [2]:
import os
import cassio
from dotenv import load_dotenv

load_dotenv()

ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_ID = os.getenv("ASTRA_DB_ID")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Connect to Astra DB
cassio.init(token=ASTRA_DB_APPLICATION_TOKEN,database_id=ASTRA_DB_ID)

print("Astra DB connection initialized successfully!")

Astra DB connection initialized successfully!


## Task 3: Load & Split PDF Document

In [5]:
from PyPDF2 import PdfReader
from langchain_text_splitters import CharacterTextSplitter

# Load the PDF file
pdf_reader = PdfReader('physical.pdf')

raw_text = ''

# Extract text from each page of the PDF
for page in pdf_reader.pages:
    content = page.extract_text()
    if content:
        raw_text += content

splitter = CharacterTextSplitter(separator='\n', chunk_size=800, chunk_overlap=200, length_function=len)

# Split the extracted text into smaller chunks
chunks = splitter.split_text(raw_text)

print("PDF loaded successfully!")
print("Total characters:", len(raw_text))
print("Total chunks:", len(chunks))

Created a chunk of size 1118, which is longer than the specified 800
Created a chunk of size 994, which is longer than the specified 800
Created a chunk of size 1600, which is longer than the specified 800


PDF loaded successfully!
Total characters: 23962
Total chunks: 41


## Task 4: Store Embeddings in AstraDB

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.vectorstores import Cassandra

# Initialize Huggingface Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create the Cassandra vector store connected to Astra DB
vector_store = Cassandra(embedding=embeddings, table_name="qa_demo_one", session=None, keyspace=None)

# 6. Insert chunks
vector_store.add_texts(chunks)
print("Embeddings stored successfully!")

c:\Users\ManishAgrahari\Desktop\my_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2231.32it/s]


Embeddings stored successfully!


In [7]:
# Verify that the stored data can be retrieved from Astra DB
query = "What information is available in the document?"

results = vector_store.similarity_search(query,k=3)

# Display the retrieved results
print("Data persistence verification")
print("Number of chunks retrieved:", len(results))

for i, doc in enumerate(results, start=1):
    print(f"\nChunk {i}:")
    print(doc.page_content)


Data persistence verification
Number of chunks retrieved: 3

Chunk 1:
2024-25
Healt H and PHysical  education  - class X4
Personality means, nature and factors affecting performance 
and transfer of training constitute one of its major components. Similarly, contents related to basic physiology, anatomy, growth and development during adolescence also form part of the content of physical education. Some other contents are organ system, functions of bones, definitions and classification of joints, movement around a joint, general characteristics (properties) of muscles, effects of exercise on muscular system, circulatory system, respiratory system and digestive system, health dimensions, environment, importance of health, health problems, hygiene, community health, aspects of school health services, food, nutrition and balanced diet, communicable

Chunk 2:
2024-25
Healt H and PHysical  education  - class X4
Personality means, nature and factors affecting performance 
and transfer of trai

## Task 5: PDF Query RAG Application

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_classic.indexes.vectorstore import VectorStoreIndexWrapper

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Initialize the Groq model
llm = ChatGroq(model="openai/gpt-oss-120b",api_key=GROQ_API_KEY)

# Connect the vector store with the index wrapper
vector_index = VectorStoreIndexWrapper(vectorstore=vector_store)

print("RAG application initialized successfully!")

# Ask questions about the PDF using the RAG application
while True:
    query = input("\nEnter your question about the PDF (or type 'quit' to exit): ").strip()

    if query.lower() == "quit":
        print("Exiting the application.")
        break

    # Skip empty questions
    if not query:
        continue

    # Retrieve relevant information and generate an answer
    answer = vector_index.query(query, llm=llm)

    print("\nAnswer:")
    print(answer)

    # Display the relevant chunks retrieved from Astra DB
    results = vector_store.similarity_search_with_score(query,k=3)

    print("\nRelevant chunks:")
    for i, (doc, score) in enumerate(results, start=1):
        print(f"\nChunk {i} | Score: {score}")
        print(doc.page_content)


RAG application initialized successfully!

Answer:
Physical Education (PE) is a branch of education that uses guided physical activities—such as gymnastics, sports, games, dance, trekking, camping, and aquatics—to develop a person’s body, mind, and spirit.  

- **Purpose:** To promote optimum physical, mental, and social development, helping individuals reach their full potential in health, fitness, motor skills, and personal/social competencies.  
- **Scope:** Includes instruction in hygiene, calisthenic exercises, athletics, and the management of various sports and outdoor activities.  

In short, PE is an educational process that combines physical activity with learning to nurture the total personality of a child or individual.

Relevant chunks:

Chunk 1 | Score: 0.920755909755405
Physical education (PE) aims at the optimum development 
of an individual through continuous process of learning and participation in guided physical activities. In other words, it aims at optimum physical

## Task 6: Testing & Validation

In [11]:
# questions to test the RAG application
test_questions = [
    "What is the main topic of the document?",
    "What are the key points discussed in the document?",
    "Explain an important concept from the document.",
    "What facts or examples are mentioned in the document?",
    "Summarize the document in simple language."
]

# Test the RAG application with each question
for i, question in enumerate(test_questions, start=1):
    print(f"Test {i}: {question}")

    # Generate an answer using the RAG index
    answer = vector_index.query(question,llm=llm)

    print("\nAnswer:")
    print(answer)
    print()

Test 1: What is the main topic of the document?

Answer:
The document’s primary focus is on an **interdisciplinary approach that links the social sciences—especially civics, geography and history—with health and physical‑education learning**. It explains how knowledge of civic institutions, map‑reading skills, and historical perspectives are integrated into health‑related studies (e.g., understanding disease burden, interpreting health data) and outdoor‑education activities, so that students develop the understanding and skills needed to participate actively in society and make evidence‑based decisions about their health and environment.

Test 2: What are the key points discussed in the document?

Answer:
**Key points from the excerpt**

1. **Purpose of Social Sciences**  
   - Emphasize that citizens need knowledge of civic institutions.  
   - Develop the skills and willingness to take an active part in society’s functioning.

2. **Health & Physical Education (HPE) – Core Themes**  


## Observations & Insights

### 1. Why AstraDB is useful for production RAG

AstraDB stores the text chunks and their embeddings in the cloud. When a user asks a question, it searches for similar chunks and returns them. The data stays stored even after restarting the application, so we don't need to insert the documents every time.

### 2. Importance of session state in GenAI apps

Session state is useful when we want to keep chat history in an application. For example, in Streamlit, we can use `st.session_state` to store previous messages so users can see their conversation even when the app reruns. It makes the chatbot easier to use for multiple questions.

### 3. Difference between FAISS and AstraDB

FAISS is useful when we want to perform vector searches locally. It is easy to use for learning and small projects. AstraDB is a cloud database that stores vectors and documents persistently. We can use FAISS for local experiments and AstraDB when we need cloud storage and want to access the data through a deployed application.

### Conclusion

In this assignment, I learned how to load a PDF, split its content into chunks, create embeddings, and store them in AstraDB. I also learned how to retrieve relevant information and use Groq to answer questions.
